In [2]:
# Country-rate layer
#
# Turns the World Bank crude death rate (deaths per 1,000 population) into a per-cell
# relative multiplier on the base grid (data/density-grid.json). This is the first
# "layer" migrated onto the grid-layer architecture: it replaces the country-level
# Poisson rate the globe used to compute at runtime (app/globe/useGlobeData.js's old
# blinkById). Region/subnational data will later replace this layer wholesale — a finer
# per-cell rate, not an additional multiplier (rates don't stack; they refine).
#
# Source: data/source/cdr-snapshot.json, produced offline by `npm run dump:cdr`
# (lib/worldbank.js's getMortality(), snapshotted because notebooks can't call the
# Next.js API route directly).

import sys
sys.path.insert(0, "../lib")
import grid

cdr = grid.load_cdr_snapshot()
print(f"{len(cdr)} countries with CDR + population.")
list(cdr.items())[:3]


169 countries with CDR + population.


[(4,
  {'iso3': 'AFG',
   'name': 'Afghanistan',
   'cdr': 5.702,
   'population': 43844111}),
 (8, {'iso3': 'ALB', 'name': 'Albania', 'cdr': 8.457, 'population': 2349580}),
 (12,
  {'iso3': 'DZA', 'name': 'Algeria', 'cdr': 4.618, 'population': 47435312})]

In [3]:
# Base grid + per-country gridded population.
#
# The country-rate layer's mean-1 value is r(m49)/r_bar, where r(m49) is the per-person
# annual death rate for that country RE-EXPRESSED over gridded (2015 GPWv4) population,
# not World Bank (2024) population. This keeps §3's identity exact: cellPop * r_bar *
# layer_value sums to CDR * WBpop / 1000 per country, regardless of which population
# vintage the grid cells carry.

base_cells, cellsize = grid.load_base()
print(f"{len(base_cells)} base-grid cells, cellsize={cellsize}")

grid_pop_by_country = {}
for c in base_cells:
    grid_pop_by_country[c["m49"]] = grid_pop_by_country.get(c["m49"], 0) + c["pop"]

print(f"{len(grid_pop_by_country)} distinct countries in the base grid.")


59863 base-grid cells, cellsize=0.5
223 distinct countries in the base grid.


In [4]:
# r(m49) = CDR * WBpop / (1000 * gridPop)   -- per-gridded-person annual death rate
# r_bar   = sum(deaths) / sum(gridPop)        -- population-weighted global mean

rate_by_country = {}
total_deaths = 0.0
total_grid_pop_with_cdr = 0.0

for m49, gpop in grid_pop_by_country.items():
    c = cdr.get(m49)
    if not c or not (gpop > 0):
        continue  # no CDR for this country (e.g. Singapore, Hong Kong) -> weight 0, handled at bake time
    deaths = c["cdr"] * c["population"] / 1000.0
    rate_by_country[m49] = deaths / gpop
    total_deaths += deaths
    total_grid_pop_with_cdr += gpop

r_bar = total_deaths / total_grid_pop_with_cdr
print(f"Countries with a usable rate: {len(rate_by_country)} / {len(grid_pop_by_country)}")
print(f"Global deaths/year (grid-covered countries): {total_deaths:,.0f}")
print(f"r_bar (deaths per gridded person per year): {r_bar:.6f}")

missing_cdr = sorted(set(grid_pop_by_country) - set(rate_by_country))
print(f"Grid countries with no CDR (weight 0): {missing_cdr}")


Countries with a usable rate: 166 / 223
Global deaths/year (grid-covered countries): 61,392,295
r_bar (deaths per gridded person per year): 0.008444
Grid countries with no CDR (weight 0): [16, 28, 48, 52, 60, 92, 132, 136, 158, 174, 175, 184, 212, 234, 238, 254, 258, 296, 308, 312, 316, 344, 462, 470, 474, 480, 500, 520, 533, 570, 574, 580, 583, 584, 585, 612, 638, 654, 659, 660, 662, 666, 670, 678, 690, 702, 744, 772, 776, 796, 798, 831, 832, 833, 850, 876, 882]


In [5]:
# Coverage check: CDR countries with NO base-grid cells at all (would silently vanish
# under a naive bake). These need synthetic cells in combine.ipynb.
cdr_countries = set(cdr.keys())
grid_countries = set(grid_pop_by_country.keys())
missing_from_grid = sorted(cdr_countries - grid_countries)
print(f"CDR countries absent from the base grid ({len(missing_from_grid)}):")
for m49 in missing_from_grid:
    c = cdr[m49]
    print(f"  {m49} {c['name']} ({c['iso3']}): CDR={c['cdr']}, pop={c['population']:,}, "
          f"deaths/yr={c['cdr']*c['population']/1000:,.0f}")


CDR countries absent from the base grid (3):
  499 Montenegro (MNE): CDR=10.2, pop=623,129, deaths/yr=6,356
  688 Serbia (SRB): CDR=14.9, pop=6,549,143, deaths/yr=97,582
  728 South Sudan (SSD): CDR=9.821, pop=12,188,788, deaths/yr=119,706


In [6]:
# get_mortality_multiplier: the notebook's authored interface. Countries with no CDR
# (Singapore, Hong Kong, ...) get exactly 1.0 here -- their weight-0 exclusion happens
# via the CDR gate in combine.ipynb (a value of 1 leaves the door open for future CDR
# coverage without redefining the layer), matching today's behavior where they never fire.

def get_mortality_multiplier(lonlat):
    lon, lat = lonlat
    m49 = grid.country_at(lon, lat)
    r = rate_by_country.get(m49)
    if r is None:
        return 1.0
    return r / r_bar

layer = grid.bake_layer(get_mortality_multiplier, "country-rate")


Wrote data/layers/country-rate.json: 59863 cells, mean=1.0000 (pre-normalize)


In [7]:
# Spot check: US (840) should read ~ r(840)/r_bar, and a Serbia/Montenegro/S.Sudan cell
# should just not exist in this layer (they have no base-grid cells at all -- handled
# separately in combine.ipynb via synthetic cells).
sample = [c for c in layer["cells"] if c[2] == 840][:3]
print("US sample cells (lon, lat, m49, value):", sample)
print("Expected US value ~", rate_by_country[840] / r_bar)


US sample cells (lon, lat, m49, value): [[-177, 51.5, 840, 1.13429], [-174.5, 52, 840, 1.13429], [-172, 63.5, 840, 1.13429]]
Expected US value ~ 1.1342899000898559
